# 🚀 Module 19 — Advanced Python: Typing, Concurrency, Async, Testing & Internals

## 🎯 What you will learn
1. Type hints & static checking (`typing`, generics, `TypedDict`, `Protocol`)
2. Memory management: reference counting, garbage collection, `weakref`
3. Concurrency: threads, processes, the GIL, `concurrent.futures`
4. `asyncio`: `async`/`await`, tasks, gather, timeouts
5. Testing: `assert`, `unittest`, `doctest`, pytest style, mocking
6. Debugging & profiling: `pdb`, `cProfile`, `timeit`, `tracemalloc`
7. Descriptors & metaclasses (the deep end)
8. Packaging & project layout; best practices
9. Hidden facts & tricks

**Exam relevance:** ⭐⭐ (advanced electives) — but this is what separates learners from engineers.

---
## 1. Type hints
Annotations document intent and enable tools (`mypy`, `pyright`, IDEs). **Python ignores them at run time.**

In [1]:
from typing import Optional, Union, Any, Callable, Iterable, Sequence, Literal, TypeAlias, TypedDict

def greet(name: str, times: int = 1) -> str:
    return (f"Hello {name}! " * times).strip()

def first(items: Sequence[int]) -> Optional[int]:      # int | None
    return items[0] if items else None

def parse(value: int | str) -> float:                   # modern union syntax (3.10+)
    return float(value)

Handler: TypeAlias = Callable[[str], None]
Mode = Literal["r", "w"]                                # only these exact values allowed by checkers

class UserDict(TypedDict):
    name: str
    age: int

def show(u: UserDict) -> str: return f"{u['name']} ({u['age']})"

nums: list[int] = [1, 2, 3]                             # built-in generics (3.9+)
mapping: dict[str, list[float]] = {"a": [1.0]}
point: tuple[int, int] = (1, 2)
print(greet("Asha", 2), first([]), parse("3.5"), show({"name": "Ravi", "age": 20}))
print(greet.__annotations__)
print(greet(123))          # no error at run time! only mypy would complain

Hello Asha! Hello Asha! None 3.5 Ravi (20)
{'name': <class 'str'>, 'times': <class 'int'>, 'return': <class 'str'>}
Hello 123!


### Generics, `TypeVar`, and `Protocol`

In [2]:
from typing import TypeVar, Generic, Protocol

T = TypeVar("T")

def head(xs: list[T]) -> T:                    # returns the SAME type it was given
    return xs[0]

class Box(Generic[T]):
    def __init__(self, item: T): self.item = item
    def get(self) -> T: return self.item

class SupportsClose(Protocol):
    def close(self) -> None: ...

def shutdown(x: SupportsClose) -> None: x.close()

print(head([1, 2]), head(["a"]), Box(3.5).get())

# Python 3.12 new syntax:  def head[T](xs: list[T]) -> T: ...   and   class Box[T]: ...
def first_or[T](xs: list[T], default: T) -> T:
    return xs[0] if xs else default
print(first_or([], "empty"))

# run "pip install mypy" then "mypy yourfile.py" to check types statically

1 a 3.5
empty


## 2. Memory management

CPython uses **reference counting** (immediate cleanup when count hits 0) plus a **cyclic garbage collector** for reference cycles.

In [3]:
import sys, gc, weakref

a = []
print(sys.getrefcount(a))          # includes the temporary reference made by the call
b = a
print(sys.getrefcount(a))
del b
print(sys.getrefcount(a))

class Node:
    def __init__(self, name): self.name, self.other = name, None
    def __del__(self): print(f"  freeing {self.name}")

x = Node("x")
del x                              # refcount 0 -> freed at once

# Reference cycle: refcounts never reach 0 -> the GC must find them
p, q = Node("p"), Node("q")
p.other, q.other = q, p
del p, q
print("collecting cycles...")
print("unreachable objects found:", gc.collect() > 0)

# weakref: a reference that doesn't keep the object alive
class Big: pass
big = Big()
ref = weakref.ref(big)
print(ref() is big)
del big
print(ref())                       # None: object already freed
print(gc.get_threshold(), gc.isenabled())

2
3
2
  freeing x
collecting cycles...
  freeing p
  freeing q
unreachable objects found: True
True
None
(2000, 10, 10) True


## 3. Concurrency: threads, processes and the GIL

- **Thread**: lightweight, shares memory. Because of the **GIL (Global Interpreter Lock)**, only one thread executes Python bytecode at a time → threads help with **I/O-bound** work (network, disk, waiting), *not* CPU-bound work.
- **Process**: separate interpreter and memory → real parallelism for **CPU-bound** work (`multiprocessing`, `ProcessPoolExecutor`).
- **asyncio**: single-threaded cooperative multitasking; thousands of concurrent I/O tasks.

| Workload | Best tool |
|---|---|
| Waiting on network/disk | `threading` / `asyncio` |
| Heavy computation | `multiprocessing` / NumPy / C extensions |
| Many thousands of connections | `asyncio` |

In [4]:
import threading, time

def worker(name, delay, results):
    time.sleep(delay)                      # simulates I/O; releases the GIL
    results.append(f"{name} done")

results = []
t0 = time.perf_counter()
threads = [threading.Thread(target=worker, args=(f"job{i}", 0.2, results)) for i in range(5)]
for t in threads: t.start()
for t in threads: t.join()                 # wait for all
print(len(results), f"finished in {time.perf_counter() - t0:.2f}s (sequential would take 1.0s)")

# Race condition and Lock
counter = 0
lock = threading.Lock()
def add_many(n):
    global counter
    for _ in range(n):
        with lock:                         # remove the lock and updates may be lost
            counter += 1
ts = [threading.Thread(target=add_many, args=(20000,)) for _ in range(4)]
[t.start() for t in ts]; [t.join() for t in ts]
print(counter)

5 finished in 0.21s (sequential would take 1.0s)
80000


### `concurrent.futures` — the easy high-level API

In [5]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

def fetch(i):
    time.sleep(0.1)
    return i * i

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    print(list(pool.map(fetch, range(10))))                       # ordered results
print(f"{time.perf_counter() - t0:.2f}s")

with ThreadPoolExecutor() as pool:
    futures = {pool.submit(fetch, i): i for i in range(3)}
    for fut in as_completed(futures):                             # in completion order
        print("task", futures[fut], "->", fut.result())

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
0.20s
task 0 -> 0
task 1 -> 1
task 2 -> 4


In [ ]:
# notest
# Processes need worker functions that can be imported by the child process. In a notebook,
# functions defined in a cell live in __main__ and CANNOT be pickled on Windows/macOS ("spawn").
# Fix: put the worker in a real module file, then import it.
import pathlib, sys
from concurrent.futures import ProcessPoolExecutor

pathlib.Path("cpu_work.py").write_text(
    "def heavy(n):\n"
    "    return sum(i * i for i in range(n))\n"
)
sys.path.insert(0, ".")
from cpu_work import heavy

with ProcessPoolExecutor() as pool:
    print(list(pool.map(heavy, [10**6] * 4)))     # runs on several CPU cores in parallel

# In a plain .py script the same idea works, but the pool must be created under:
#     if __name__ == "__main__":

## 4. `asyncio` — async / await

`async def` defines a **coroutine**; `await` pauses it until the awaited thing is ready, letting other tasks run. One thread, one event loop.

In [6]:
import asyncio

def run_async(coro):
    """Run a coroutine from a script OR from a notebook (where an event loop already runs)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)                       # plain script: no loop yet
    from concurrent.futures import ThreadPoolExecutor  # notebook: use a fresh loop in a helper thread
    with ThreadPoolExecutor(1) as ex:
        return ex.submit(asyncio.run, coro).result()

async def fetch_data(name, delay):
    print(f"  start {name}")
    await asyncio.sleep(delay)            # non-blocking wait
    print(f"  end   {name}")
    return f"{name}:{delay}"

async def main():
    t0 = time.perf_counter()
    results = await asyncio.gather(fetch_data("A", 0.3), fetch_data("B", 0.2), fetch_data("C", 0.1))
    print(results, f"{time.perf_counter() - t0:.2f}s (concurrent!)")

    # Tasks start immediately; timeouts; exceptions
    task = asyncio.create_task(fetch_data("T", 0.1))
    print(await task)
    try:
        await asyncio.wait_for(fetch_data("slow", 2), timeout=0.2)
    except asyncio.TimeoutError:
        print("  timed out!")

    # Producer / consumer with a queue
    q = asyncio.Queue()
    async def producer():
        for i in range(3): await q.put(i); await asyncio.sleep(0.01)
        await q.put(None)
    async def consumer():
        while (item := await q.get()) is not None: print("  consumed", item)
    await asyncio.gather(producer(), consumer())

    # async context managers / iterators exist too: async with, async for
    async def ticker(n):
        for i in range(n):
            await asyncio.sleep(0.01); yield i
    print([i async for i in ticker(3)])

run_async(main())          # in a plain script this is just asyncio.run(main()); in a notebook you may also write:  await main()

  start A
  start B
  start C
  end   C
  end   B
  end   A
['A:0.3', 'B:0.2', 'C:0.1'] 0.31s (concurrent!)
  start T
  end   T
T:0.1
  start slow
  timed out!
  consumed 0
  consumed 1
  consumed 2
[0, 1, 2]


> Rule: never call blocking functions (`time.sleep`, `requests.get`) inside a coroutine — use `await asyncio.sleep`, `aiohttp`/`httpx`, or `asyncio.to_thread(func)`.

## 5. Testing

### 5.1 `assert` and pytest-style tests
pytest discovers files named `test_*.py` and functions named `test_*`.

In [7]:
def is_prime(n: int) -> bool:
    if n < 2: return False
    return all(n % d for d in range(2, int(n ** 0.5) + 1))

def test_primes():
    assert is_prime(2) and is_prime(97)
    assert not is_prime(1) and not is_prime(0) and not is_prime(91)

def test_negative():
    assert not is_prime(-7)

test_primes(); test_negative()
print("all asserts passed")

all asserts passed


### 5.2 `unittest` (built in)

In [8]:
import unittest

class TestPrime(unittest.TestCase):
    def setUp(self):
        self.primes = [2, 3, 5, 7]
    def test_primes(self):
        for p in self.primes:
            self.assertTrue(is_prime(p), msg=f"{p} should be prime")
    def test_composite(self):
        self.assertFalse(is_prime(9))
    def test_type_error(self):
        with self.assertRaises(TypeError):
            is_prime("x")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestPrime)
result = unittest.TextTestRunner(verbosity=0).run(suite)
print("passed:", result.wasSuccessful(), "| ran:", result.testsRun)
# From a terminal: python -m unittest -v      or      pytest

passed: True | ran: 3


----------------------------------------------------------------------
Ran 3 tests in 0.000s

OK


### 5.3 `doctest` — tests inside docstrings

In [9]:
import doctest

def cube(x):
    """Return x cubed.

    >>> cube(3)
    27
    >>> cube(-2)
    -8
    >>> [cube(i) for i in range(4)]
    [0, 1, 8, 27]
    """
    return x ** 3

print(doctest.run_docstring_examples(cube, globals(), verbose=False) or "doctest ok")

doctest ok


### 5.4 Mocking — replace slow/unreliable dependencies

In [10]:
from unittest.mock import Mock, patch, MagicMock

def get_weather(client, city):
    data = client.fetch(city)
    return f"{city}: {data['temp']}°C"

fake = Mock()
fake.fetch.return_value = {"temp": 31}
print(get_weather(fake, "Pune"))
fake.fetch.assert_called_once_with("Pune")

import random
with patch("random.random", return_value=0.5):        # temporarily replace a function
    print(random.random())
print(0 <= random.random() < 1)

Pune: 31°C
0.5
True


### Testing best practices
- **AAA**: Arrange, Act, Assert. One idea per test. Tests must be fast and independent.
- Test edge cases: empty, one item, negatives, huge, wrong types.
- **TDD**: write a failing test first, then code, then refactor.
- Measure coverage: `pip install pytest-cov` → `pytest --cov`.

## 6. Debugging & profiling

In [11]:
import cProfile, pstats, io, timeit, tracemalloc

def slow():
    total = 0
    for i in range(200_000):
        total += i % 7
    return sorted(str(i) for i in range(20_000))

pr = cProfile.Profile(); pr.enable(); slow(); pr.disable()
buf = io.StringIO()
pstats.Stats(pr, stream=buf).sort_stats("cumulative").print_stats(3)
print("\n".join(buf.getvalue().splitlines()[:12]))

print(timeit.repeat("sum(range(1000))", number=2000, repeat=3)[0] > 0)

tracemalloc.start()
data = [str(i) * 10 for i in range(20_000)]
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"memory in use: {current / 1e6:.1f} MB, peak {peak / 1e6:.1f} MB")

         20004 function calls in 0.020 seconds

   Ordered by: cumulative time
   List reduced from 4 to 3 due to restriction <3>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.012    0.012    0.020    0.020 <19_Advanced_Python#22>:3(slow)
        1    0.003    0.003    0.008    0.008 {built-in method builtins.sorted}
    20001    0.005    0.000    0.005    0.000 <19_Advanced_Python#22>:7(<genexpr>)


True
memory in use: 1.9 MB, peak 1.9 MB


Debugging toolbox: `print`/f-string `{x=}`, `logging`, `breakpoint()` (pdb: `n s c p l q`), VS Code debugger (breakpoints, watch, call stack), `traceback`, `python -X dev`, `python -m trace`.

## 7. Descriptors & metaclasses (the deep end)

**Descriptors** power `property`, methods, `classmethod`, `__slots__`: any object with `__get__`/`__set__`/`__delete__` stored as a class attribute.

In [12]:
class Positive:
    """A reusable validated attribute."""
    def __set_name__(self, owner, name): self.private = "_" + name
    def __get__(self, obj, objtype=None):
        return self if obj is None else getattr(obj, self.private)
    def __set__(self, obj, value):
        if value <= 0: raise ValueError(f"{self.private[1:]} must be positive")
        setattr(obj, self.private, value)

class Product:
    price = Positive()
    qty = Positive()
    def __init__(self, price, qty): self.price, self.qty = price, qty

p = Product(9.5, 3)
print(p.price, p.qty)
try:
    p.qty = -1
except ValueError as e:
    print("ValueError:", e)

9.5 3
ValueError: qty must be positive


**Metaclasses**: a class is an object; its class (the *metaclass*) is by default `type`. `class Foo` is roughly `Foo = type("Foo", bases, namespace)`.

In [13]:
class AutoRegister(type):
    registry = {}
    def __new__(mcls, name, bases, ns):
        cls = super().__new__(mcls, name, bases, ns)
        mcls.registry[name] = cls
        return cls

class Base(metaclass=AutoRegister): pass
class Alpha(Base): pass
class Beta(Base): pass
print(list(AutoRegister.registry), type(Alpha), type(type))
# In practice prefer __init_subclass__ or class decorators; metaclasses are rarely needed.

['Base', 'Alpha', 'Beta'] <class '__main__.AutoRegister'> <class 'type'>


## 8. Packaging & project layout

```
my_project/
├── pyproject.toml         # metadata, dependencies, build config (modern standard)
├── README.md
├── src/
│   └── mypkg/
│       ├── __init__.py
│       ├── core.py
│       └── cli.py
├── tests/
│   └── test_core.py
└── .gitignore
```

Minimal `pyproject.toml`:

```toml
[project]
name = "mypkg"
version = "0.1.0"
dependencies = ["requests>=2"]
[project.scripts]
mycli = "mypkg.cli:main"
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"
```

Commands: `pip install -e .` (editable install), `python -m build`, `twine upload dist/*`.

Tooling that professionals use: **ruff** (lint+format), **black**, **mypy/pyright**, **pytest**, **pre-commit**, **uv** (fast installer), **git**, **Docker**, **GitHub Actions** (CI).

## 9. Best practices checklist ✅
1. Follow PEP 8 (auto-format with `ruff format`/`black`).
2. Write docstrings and type hints for public functions.
3. Small functions doing one thing; avoid globals and mutable defaults.
4. Handle errors specifically; log, don't print.
5. Use `pathlib`, `with`, f-strings, comprehensions, `enumerate`, `zip`.
6. Write tests; automate with CI.
7. Pin dependencies in a virtual environment.
8. Never commit secrets — use environment variables (`os.environ["API_KEY"]`, `.env` files ignored by git).
9. Prefer readability over cleverness ("Readability counts").
10. Profile before optimising.

---
# 🕵️ Hidden Facts & Pro Tricks

In [14]:
# 1. Python 3.13 introduced an optional free-threaded build (no GIL) and an experimental JIT
import sys
print(getattr(sys, "_is_gil_enabled", lambda: True)())

# 2. asyncio.to_thread runs blocking code without freezing the event loop
async def demo():
    return await asyncio.to_thread(sum, range(1000))
print(run_async(demo()))

# 3. TaskGroup (3.11+): structured concurrency, errors cancel siblings
async def tg():
    async with asyncio.TaskGroup() as g:
        t1 = g.create_task(fetch_data("x", 0.01)); t2 = g.create_task(fetch_data("y", 0.01))
    return t1.result(), t2.result()
print(run_async(tg()))

# 4. functools.cached_property: compute once per instance
from functools import cached_property
class Data:
    @cached_property
    def heavy(self):
        print("  computing once"); return sum(range(10**5))
d = Data(); d.heavy; print(d.heavy)

# 5. contextvars, threading.local: per-thread/per-task state
# 6. `python -X importtime script.py` shows slow imports; `python -m cProfile -s cumtime script.py`
# 7. `__debug__`, `python -O` strips asserts; never use assert for validation of user input
# 8. `sys.intern`, `slots=True` dataclasses, and `array` reduce memory for millions of objects
from dataclasses import dataclass
@dataclass(slots=True)
class Pt: x: int; y: int
print(sys.getsizeof(Pt(1, 2)))

# 9. `typing.assert_never`, `@override` (3.12), `@final`, `Self`, `ParamSpec` improve type safety
from typing import Self
class Builder:
    def set(self, k: str) -> Self: return self

# 10. Structural pattern matching works on dataclasses and dicts
@dataclass
class Click: x: int; y: int
match Click(1, 2):
    case Click(x=1, y=yy): print("clicked at column 1, y =", yy)

# 11. The `dis` module shows why `a, b = b, a` is fast: no temp variable, just ROT/SWAP opcodes
# 12. Interpreter flags: -m (run module), -i (interactive after), -O, -B (no .pyc), -X dev

True
499500
  start x
  start y
  end   x
  end   y
('x:0.01', 'y:0.01')
  computing once
4999950000
48
clicked at column 1, y = 2


**Hidden fact:** the GIL exists because CPython's reference counting isn't thread-safe. Removing it (PEP 703) means atomic refcounts — which is why the free-threaded build is opt-in and single-thread performance dips slightly.

**Hidden fact:** `async` code isn't automatically faster — it only helps when there is *waiting*. CPU-bound work inside a coroutine blocks every other task.

**Hidden fact:** Python releases a new version every October; each is supported for 5 years. The Steering Council (5 elected members) replaced the "BDFL" model when Guido stepped down in 2018.

---
# 📝 Exam Corner

**Theory**
1. What is the GIL? Threading vs multiprocessing — when to use which?
2. Explain reference counting and garbage collection.
3. What are type hints? Are they enforced?
4. Explain `async`/`await` and the event loop.
5. What is unit testing? Explain `unittest` with an example.
6. What is a virtual environment; what is `pyproject.toml`?

**Output prediction**

In [15]:
import asyncio
async def a():
    print("a1"); await asyncio.sleep(0); print("a2")
async def b():
    print("b1"); await asyncio.sleep(0); print("b2")
async def m():
    await asyncio.gather(a(), b())
run_async(m())

def f(x: int) -> str:
    return x * 2
print(f(4), type(f(4)))

a1
b1
a2
b2
8 <class 'int'>


<details><summary>💡 Answers</summary>

```
a1
b1
a2
b2
8 <class 'int'>
```
Coroutines interleave at each `await`. The hint says `-> str` but nothing enforces it at run time.
</details>

---
## ✅ Summary
- Type hints = documentation + tooling, not enforcement. Generics/Protocols describe structure.
- Refcount + cyclic GC manage memory; `weakref` avoids keeping objects alive.
- Threads for I/O (GIL blocks CPU parallelism), processes for CPU, `asyncio` for massive I/O concurrency.
- Test with pytest/unittest/doctest + mocks; profile before optimising.

➡️ **Next:** Module 20 — Python Secrets, Tricks & One-Liners (the fun one), then the `challenges/` and `exam_prep/` folders.